#Ingest - Race Data & Race Events - _James Trotman_
> Gets all the race data & race events CSVs from s3 for Race Data

In [0]:
%run /Workspace/Repos/nikum.vedansh@gmail.com/formula-one-project/configs/credentials

In [0]:
dbutils.widgets.text("schema", "default", "Schema")

paths = [RACE_DATA_PATH, RACE_EVENTS_PATH]

succeeded, failed, skipped = [], [], []

In [0]:
schema = dbutils.widgets.get("schema")

In [0]:
import re
import time
from datetime import datetime
from multiprocessing.pool import ThreadPool

In [0]:
invalid_chars_pattern = re.compile(r"[ ,;{}()\n\t=]")

def sanitize_columns(df):
    return df.toDF(*[invalid_chars_pattern.sub("_", c.lower()) for c in df.columns])

def process_file(args):
    file, folder_name = args
    clean_name = re.sub(r"[^a-zA-Z0-9_]", "_", file.name.replace('.csv', ''))
    table_ident = f"{folder_name}_{clean_name}"
    full_table_path = f"formone.{schema}.{table_ident}"

    if spark.catalog.tableExists(full_table_path):
        return ("skipped", file.name, None)

    try:
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("dateFormat", "dd/MM/yyyy") \
            .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
            .load(file.path)

        df = sanitize_columns(df)
        df.write.format("delta").mode("overwrite").saveAsTable(full_table_path)  # dropped coalesce
        return ("succeeded", file.name, None)

    except Exception as e:
        return ("failed", file.name, f"{type(e).__name__}: {str(e)[:300]}")

In [0]:
# --- Build file list ---
all_files = []
for path in paths:
    folder_name = path.rstrip("/").split("/")[-1]
    try:
        for f in dbutils.fs.ls(path):
            if f.name.endswith(".csv"):
                all_files.append((f, folder_name))
    except Exception as e:
        print(f"  ❌ Could not list directory {path}: {e}")

In [0]:
succeeded, failed, skipped = [], [], []

In [0]:
# --- Run ---
start_time = time.time()
print("=" * 40)
print(f"  Bronze Ingestion Started")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 40 + "\n")

with ThreadPool(9) as pool:
    for status, name, reason in pool.map(process_file, all_files):
        if status == "succeeded":
            print(f"  ✅ {name}")
            succeeded.append(name)
        elif status == "skipped":
            print(f"  ⏭️  {name}")
            skipped.append(name)
        else:
            print(f"  ❌ {name}")
            print(f"     {reason}")
            failed.append((name, reason))

# --- Final Summary ---
total_duration = round(time.time() - start_time, 2)

print("\n" + "=" * 40)
print(f"  Finished — {total_duration}s")
print("=" * 40)

print(f"\n  ✅ Succeeded ({len(succeeded)})")
for name in succeeded:
    print(f"     - {name}")

print(f"\n  ⏭️  Skipped ({len(skipped)})")
for name in skipped:
    print(f"     - {name}")

print(f"\n  ❌ Failed ({len(failed)})")
if failed:
    for name, reason in failed:
        print(f"     - {name}")
        print(f"       {reason}")
else:
    print("     None")

print("\n" + "=" * 40)